# NSE data inspection

Load market data into pandas DataFrames for ad-hoc exploration. Analysis cells will be added as needed.

- **Stocks:** all `nse_cm_*.parquet` files under `PARQUET_DATA_DIR` → `df`
- **NIFTY 50 index:** `NIFTY50.parquet` → `nifty_df` (when present)

Override data root with env `QUANT_DATA_DIR` (see `quant.config`).

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import importlib.util

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "quant").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

for pkg in ("pandas", "pyarrow", "duckdb"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import duckdb
import pandas as pd

from quant.config import DATA_DIR, PARQUET_DATA_DIR, NIFTY50_PATH

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

print("DATA_DIR:", DATA_DIR)
print("PARQUET_DATA_DIR:", PARQUET_DATA_DIR, "exists:", PARQUET_DATA_DIR.exists())
print("NIFTY50_PATH:", NIFTY50_PATH, "exists:", NIFTY50_PATH.exists())

In [ ]:
def stock_parquet_files(data_dir: Path) -> list[Path]:
    return sorted(
        p
        for p in data_dir.rglob("nse_cm_*.parquet")
        if "results" not in p.parts and "indices" not in p.parts
    )


def load_all_stock_bars(files: list[Path], *, data_dir: Path) -> pd.DataFrame:
    """Load every daily stock parquet into one DataFrame (DuckDB union)."""
    if not files:
        raise FileNotFoundError(f"No nse_cm_*.parquet under {data_dir}")
    con = duckdb.connect()
    paths_sql = ", ".join(f"'{str(p).replace(chr(39), chr(39) * 2)}'" for p in files)
    sql = f"""
    SELECT *
    FROM read_parquet([{paths_sql}], union_by_name=true)
    """
    out = con.execute(sql).df()
    cols = {c: c.lower().strip() for c in out.columns}
    out = out.rename(columns=cols)
    date_col = next((c for c in ("date", "datetime", "timestamp") if c in out.columns), None)
    if date_col:
        out[date_col] = pd.to_datetime(out[date_col], errors="coerce")
    sym_col = next((c for c in ("symbol", "ticker", "tradingsymbol") if c in out.columns), None)
    if sym_col and sym_col != "symbol":
        out = out.rename(columns={sym_col: "symbol"})
    if "symbol" in out.columns:
        out["symbol"] = out["symbol"].astype(str).str.upper().str.strip()
    return out


files = stock_parquet_files(PARQUET_DATA_DIR)
print(f"Found {len(files)} stock parquet files")
if files:
    print("First:", files[0].name, "| Last:", files[-1].name)

In [ ]:
# Full stock universe (may be large — ensure sufficient RAM)
df = load_all_stock_bars(files, data_dir=PARQUET_DATA_DIR)
print("df shape:", df.shape)
print("Columns:", list(df.columns))
date_col = "date" if "date" in df.columns else None
if date_col:
    print("Date range:", df[date_col].min(), "→", df[date_col].max())
if "symbol" in df.columns:
    print("Symbols:", df["symbol"].nunique())
df.head()

In [ ]:
nifty_df = None
if NIFTY50_PATH.exists():
    nifty_df = pd.read_parquet(NIFTY50_PATH)
    nifty_df.columns = [c.lower().strip() for c in nifty_df.columns]
    if "date" in nifty_df.columns:
        nifty_df["date"] = pd.to_datetime(nifty_df["date"], errors="coerce")
    print("nifty_df shape:", nifty_df.shape)
    display(nifty_df.head())
else:
    print("NIFTY50 parquet not found — skip index series.")

## Optional: pre-built explosive-moves panel

If you already ran `explosive_moves_research.ipynb` Phase 2, you can load the stitched panel instead of re-reading every source file:

In [ ]:
PANEL_PATH = DATA_DIR / "results" / "explosive_moves" / "panel" / "ohlcv_panel.parquet"
panel_df = None
if PANEL_PATH.exists():
    panel_df = pd.read_parquet(PANEL_PATH)
    panel_df["Date"] = pd.to_datetime(panel_df["Date"])
    print("panel_df:", panel_df.shape, panel_df["Date"].min(), "→", panel_df["Date"].max())
    display(panel_df.head())
else:
    print("No stitched panel at", PANEL_PATH)

## Data gaps

Detect missing **exchange sessions** (not just calendar days): compare each symbol's bars to a reference session calendar (NIFTY when available, else breadth-filtered panel union).

Outputs:
- `market_calendar_gaps` — reference sessions with thin / missing panel coverage
- `symbol_session_gaps` — each internal missing session run per symbol
- `symbol_gap_summary` — rolled-up counts per symbol
- `duplicate_bars`, `weekend_bars` — basic QA

In [ ]:
import numpy as np

from quant.research.explosive_research import (
    build_exchange_session_registry,
    refined_trading_calendar,
)


def _pick_bars_frame() -> pd.DataFrame:
    if "panel_df" in globals() and panel_df is not None:
        return panel_df.copy()
    if "df" in globals() and df is not None:
        return df.copy()
    raise RuntimeError("Load df (raw) or panel_df first.")


def _canonical_ohlcv(raw: pd.DataFrame) -> pd.DataFrame:
    x = raw.copy()
    colmap = {c: c.lower().strip() for c in x.columns}
    x = x.rename(columns=colmap)
    date_col = next((c for c in ("date", "datetime", "timestamp") if c in x.columns), None)
    sym_col = next((c for c in ("symbol", "ticker", "tradingsymbol") if c in x.columns), None)
    if date_col is None or sym_col is None:
        raise ValueError(f"Need date + symbol columns; got {list(x.columns)}")
    out = pd.DataFrame(
        {
            "Date": pd.to_datetime(x[date_col], errors="coerce"),
            "Symbol": x[sym_col].astype(str).str.upper().str.strip(),
        }
    )
    for src, dst in (
        ("open", "Open"),
        ("high", "High"),
        ("low", "Low"),
        ("close", "Close"),
        ("volume", "Volume"),
        ("turnover", "Turnover"),
    ):
        if src in x.columns:
            out[dst] = pd.to_numeric(x[src], errors="coerce")
    out = out.dropna(subset=["Date", "Symbol"])
    return out.sort_values(["Symbol", "Date"]).reset_index(drop=True)


def _nifty_for_registry(nifty) -> pd.DataFrame | None:
    if nifty is None or not isinstance(nifty, pd.DataFrame) or nifty.empty:
        return None
    y = nifty.copy()
    y.columns = [c.lower().strip() for c in y.columns]
    date_col = next((c for c in ("date", "datetime") if c in y.columns), None)
    close_col = next((c for c in ("close", "adj_close") if c in y.columns), None)
    if not date_col or not close_col:
        return None
    return pd.DataFrame(
        {
            "Date": pd.to_datetime(y[date_col], errors="coerce"),
            "Close": pd.to_numeric(y[close_col], errors="coerce"),
            "Open": pd.to_numeric(y.get("open", y[close_col]), errors="coerce"),
            "High": pd.to_numeric(y.get("high", y[close_col]), errors="coerce"),
            "Low": pd.to_numeric(y.get("low", y[close_col]), errors="coerce"),
            "Volume": pd.to_numeric(y.get("volume", 0), errors="coerce"),
        }
    ).dropna(subset=["Date", "Close"])


def find_symbol_session_gaps(bars: pd.DataFrame, ref_cal: pd.DatetimeIndex) -> pd.DataFrame:
    """Missing reference sessions strictly between two consecutive bars for a symbol."""
    ref = pd.DatetimeIndex(pd.to_datetime(ref_cal))
    rows: list[dict] = []
    for sym, g in bars.groupby("Symbol", observed=True):
        dts = pd.to_datetime(g["Date"].drop_duplicates().sort_values())
        pos = ref.get_indexer(dts)
        pos = pos[pos >= 0]
        if len(pos) < 2:
            continue
        pos = np.sort(pos)
        for i in range(len(pos) - 1):
            missing = pos[i + 1] - pos[i] - 1
            if missing <= 0:
                continue
            gap_start = ref[pos[i]]
            gap_end = ref[pos[i + 1]]
            missing_dates = list(ref[pos[i] + 1 : pos[i + 1]])
            rows.append(
                {
                    "Symbol": sym,
                    "bar_before": gap_start,
                    "bar_after": gap_end,
                    "missing_sessions": int(missing),
                    "first_missing_session": missing_dates[0],
                    "last_missing_session": missing_dates[-1],
                }
            )
    return pd.DataFrame(rows)


# Optional — limit analysis window (clip NIFTY to the same range)
BARS_START = None  # e.g. pd.Timestamp("2019-01-01")
BARS_END = None  # e.g. pd.Timestamp("2019-12-31")

bars = _canonical_ohlcv(_pick_bars_frame())
if BARS_START is not None:
    bars = bars.loc[bars["Date"] >= pd.Timestamp(BARS_START)]
if BARS_END is not None:
    bars = bars.loc[bars["Date"] <= pd.Timestamp(BARS_END)]

nifty_panel = _nifty_for_registry(globals().get("nifty_df"))
if nifty_panel is not None and not bars.empty:
    dmin, dmax = bars["Date"].min(), bars["Date"].max()
    nifty_panel = nifty_panel.loc[(nifty_panel["Date"] >= dmin) & (nifty_panel["Date"] <= dmax)]

session_registry = build_exchange_session_registry(bars, nifty_panel)
REF_CAL = refined_trading_calendar(session_registry)
PANEL_CAL = pd.DatetimeIndex(sorted(bars["Date"].unique()))

print("Bars:", bars.shape, "symbols:", bars["Symbol"].nunique())
print("Panel union sessions:", len(PANEL_CAL), "| Reference sessions:", len(REF_CAL))
print("Reference:", "NIFTY-backed" if nifty_panel is not None else "breadth-filtered panel union")

In [ ]:
missing_vol = pd.DataFrame()
if "Volume" in bars.columns:
    missing_vol = bars.loc[bars["Volume"].isna() | (bars["Volume"] <= 0), ["Date", "Symbol", "Volume"]]

duplicate_bars = (
    bars.groupby(["Date", "Symbol"], observed=True)
    .size()
    .reset_index(name="n_rows")
    .query("n_rows > 1")
    .sort_values("n_rows", ascending=False)
)

weekend_bars = bars.loc[bars["Date"].dt.weekday >= 5, ["Date", "Symbol"]].drop_duplicates()

market_calendar_gaps = session_registry.loc[~session_registry["is_valid_session"]].copy()
daily_symbol_count = bars.groupby("Date", observed=True)["Symbol"].nunique()
MKT_WIDE_FLOOR = 200  # below → treat as market-wide thin / missing day, not one symbol

if nifty_panel is not None:
    nifty_dates = set(pd.to_datetime(nifty_panel["Date"].unique()))
    panel_dates = set(PANEL_CAL)
    missing_nifty_in_panel = sorted(nifty_dates - panel_dates)
    extra_panel_not_nifty = sorted(panel_dates - nifty_dates)
else:
    missing_nifty_in_panel, extra_panel_not_nifty = [], []

symbol_session_gaps = find_symbol_session_gaps(bars, REF_CAL)
symbol_gap_summary = pd.DataFrame()
if not symbol_session_gaps.empty:
    symbol_gap_summary = (
        symbol_session_gaps.groupby("Symbol", observed=True)
        .agg(
            n_gaps=("missing_sessions", "size"),
            total_missing_sessions=("missing_sessions", "sum"),
            max_gap_sessions=("missing_sessions", "max"),
            first_gap=("first_missing_session", "min"),
            last_gap=("last_missing_session", "max"),
        )
        .sort_values("total_missing_sessions", ascending=False)
        .reset_index()
    )

print("Duplicate (Date, Symbol):", len(duplicate_bars))
print("Weekend rows:", len(weekend_bars))
print("Zero/NA volume rows:", len(missing_vol))
dmin, dmax = bars["Date"].min(), bars["Date"].max()
print("Analysis window:", dmin.date() if pd.notna(dmin) else "?", "→", dmax.date() if pd.notna(dmax) else "?")
print("Symbol session gap episodes:", len(symbol_session_gaps))
print("Symbols with ≥1 session gap:", symbol_gap_summary.shape[0] if not symbol_gap_summary.empty else 0)
if missing_nifty_in_panel:
    print("NIFTY sessions in window absent from panel union:", len(missing_nifty_in_panel))
if extra_panel_not_nifty:
    print("Panel union dates in window with no NIFTY bar:", len(extra_panel_not_nifty))

gap_date_rows: list[dict] = []
if not symbol_session_gaps.empty:
        for _, ep in symbol_session_gaps.iterrows():
            i0 = REF_CAL.get_indexer([pd.Timestamp(ep["first_missing_session"])])[0]
            i1 = REF_CAL.get_indexer([pd.Timestamp(ep["last_missing_session"])])[0]
            if i0 < 0 or i1 < 0:
                continue
            for d in REF_CAL[i0 : i1 + 1]:
                d = pd.Timestamp(d)
                n_sym = int(daily_symbol_count.get(d, 0))
                gap_date_rows.append(
                    {"Date": d, "Symbol": ep["Symbol"], "n_symbols_that_day": n_sym}
                )
        gap_by_date = pd.DataFrame(gap_date_rows)
        if not gap_by_date.empty:
            day_kind = (
                gap_by_date.groupby("Date", observed=True)["n_symbols_that_day"]
                .max()
                .reset_index()
            )
            day_kind["kind"] = np.where(
                day_kind["n_symbols_that_day"] < MKT_WIDE_FLOOR,
                "market_wide_or_holiday",
                "symbol_specific",
            )
            n_mkt = int((day_kind["kind"] == "market_wide_or_holiday").sum())
            n_sym = int((day_kind["kind"] == "symbol_specific").sum())
            print(
                "Distinct missing session dates in symbol gaps:",
                len(day_kind),
                f"(market-wide/thin: {n_mkt}, symbol-specific: {n_sym})",
            )
        symbol_session_gaps = symbol_session_gaps.assign(
            gap_class=np.where(
                symbol_session_gaps["missing_sessions"] >= 5,
                "likely_suspension_or_listing",
                "short_absence",
            )
        )

display(market_calendar_gaps.head(15))
if not symbol_gap_summary.empty:
    display(symbol_gap_summary.head(20))
if not symbol_session_gaps.empty:
    display(symbol_session_gaps.nlargest(15, "missing_sessions"))

In [ ]:
INSPECT_DIR = DATA_DIR / "results" / "inspection"
INSPECT_DIR.mkdir(parents=True, exist_ok=True)

session_registry.to_parquet(INSPECT_DIR / "session_registry.parquet", index=False)
market_calendar_gaps.to_parquet(INSPECT_DIR / "market_calendar_gaps.parquet", index=False)
symbol_session_gaps.to_parquet(INSPECT_DIR / "symbol_session_gaps.parquet", index=False)
symbol_gap_summary.to_parquet(INSPECT_DIR / "symbol_gap_summary.parquet", index=False)
duplicate_bars.to_parquet(INSPECT_DIR / "duplicate_bars.parquet", index=False)
weekend_bars.to_parquet(INSPECT_DIR / "weekend_bars.parquet", index=False)
print("Wrote gap reports to", INSPECT_DIR)